# LlamaIndex：以数据和检索为中心

LlamaIndex 最强的部分不是通用 Agent，而是：

`Document → Node → Index → Retriever → QueryEngine → 响应合成`

适合企业知识库、PDF/网页问答、产品手册、合同检索、论文库。

> 项目文件名沿用现有的 `Llamalnd测试.ipynb`；内容对应 LlamaIndex。

官方文档：https://developers.llamaindex.ai/python/framework/

项目对应实现：`support_project/adapters/llamaindex_adapter.py`


In [ ]:
from pathlib import Path
from langchain_cases.common import setup_utf8
from langchain_cases.config import get_settings

setup_utf8()

s = get_settings()
print("模型:", s.chat_model)


## 1. 建立政策索引并查询

这里把售后政策按小节切成 `Document`，使用本地哈希向量建立
`VectorStoreIndex`，再通过 `QueryEngine` 做“检索 + 回答”。

当前百炼 Key 没有 Embedding 权限，因此生产环境应替换成真实 embedding。

In [ ]:
from llama_index.core import Document, VectorStoreIndex
from support_project.domain import policy_sections
from support_project.local_embedding import HashingLlamaEmbedding
from support_project.adapters.llamaindex_adapter import _build_llm

documents = [
    Document(text=f"{x['title']}\n{x['content']}", metadata={"title": x["title"]})
    for x in policy_sections()
]
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=HashingLlamaEmbedding(),
    transformations=[],
    show_progress=False,
)
query_engine = index.as_query_engine(llm=_build_llm(), similarity_top_k=2, response_mode="compact")
response = query_engine.query("数码商品已经拆封，但出现质量问题，可以退款吗？")
print(response)
print("\n来源：", [node.metadata.get("title") for node in response.source_nodes])


## 1.1 可解释性：查看检索来源和分数

RAG 不能只返回一段答案，还要能回答“依据是什么”。LlamaIndex 的
`Response.source_nodes` 会保留命中的节点、来源 metadata 和相似度分数。

In [ ]:
for i, node in enumerate(response.source_nodes, start=1):
    score = getattr(node, "score", None)
    score_text = f"{score:.4f}" if isinstance(score, (int, float)) else "n/a"
    print(
        f"[{i}] title={node.metadata.get('title')} "
        f"score={score_text} "
        f"text={node.text[:80]}..."
    )


## 2. 售后项目：QueryEngineTool + FunctionAgent

`QueryEngineTool` 把“检索 + 生成”包装成工具，`FunctionAgent` 再决定何时调用它。

In [ ]:
from support_project.adapters.llamaindex_adapter import run

result = await run("订单 A1001 的降噪耳机坏了，想退款，是否符合政策？")
print("耗时：", round(result.elapsed_ms / 1000, 2), "s")
print("策略：", result.strategy)
print("\n", result.answer)


## 何时选 LlamaIndex

- 核心问题是“把大量文档变成可查询、可引用的知识库”；
- 需要数据连接器、索引、Retriever、QueryEngine；
- 需要把 RAG 能力作为独立工具嵌入其他 Agent 框架。

不太适合：以业务 API 调用和多工具 Agent 为核心的场景。

项目入口：

```powershell
uv run python -m support_project.runner --framework llamaindex --full
```
